In [ ]:
import jax.numpy as jnp
import numpy as np
import time
import matplotlib.pyplot as plt
from desc import set_device
set_device("gpu")
from desc.equilibrium import Equilibrium
from desc.objectives import QuadcoilFreeBoundaryError, QuadcoilProxy
from desc.plotting import *
from desc.grid import LinearGrid
from quadcoil.quantity import f_B, B2_self
from shared import quadcoil_kwargs_basic, quadcoil_kwargs_nescoil, vacuum, B2_self_target, qa_eq

/home/lf2869/Documents/Codes/DESC/desc/__init__.py:94: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


mem
Building objective: force
Precomputing transforms
Timer: Precomputing transforms = 691 ms
Timer: Objective build = 921 ms
Building objective: lcfs R
Building objective: lcfs Z
Building objective: fixed Psi
Building objective: fixed pressure
Building objective: fixed current
Building objective: fixed sheet current
Building objective: self_consistency R
Building objective: self_consistency Z
Building objective: lambda gauge
Building objective: axis R self consistency
Building objective: axis Z self consistency
Timer: Objective build = 490 ms
Timer: LinearConstraintProjection build = 10.0 sec
Number of parameters: 6480
Number of objectives: 37570
Timer: Initializing the optimization = 15.1 sec

Starting optimization
Using method: lsq-exact
Solver options:
------------------------------------------------------------
Maximum Function Evaluations       : 501
Maximum Allowed Total Δx Norm      : inf
Scaled Termination                 : True
Trust Region Method                : qr
Initial 

2026-08-27 09:33:52.585751: W external/xla/xla/hlo/transforms/simplifiers/hlo_rematerialization.cc:3023] Can't reduce memory use below 3.40GiB (3655370510 bytes) by rematerialization; only reduced to 3.94GiB (4231284512 bytes), down from 3.94GiB (4231284512 bytes) originally


       1              4          1.230e-02      6.954e-02      3.347e-01      6.945e-02   
       2              5          2.418e-04      1.205e-02      1.800e-01      5.141e-03   
       3              6          6.934e-06      2.349e-04      5.064e-02      6.591e-04   
       4              8          8.908e-07      6.044e-06      5.207e-02      2.135e-04   
       5             10          1.153e-07      7.755e-07      2.841e-02      1.096e-04   
       6             12          2.848e-08      8.685e-08      1.919e-02      9.164e-05   


In [ ]:
eq_fbe = Equilibrium.load('data/f_max_B2_eq_15.h5.h5')
eq_qss = Equilibrium.load('data_fixed/f_max_B2_eq_9.h5')
eq_ori = qa_eq

In [ ]:
quadcoil_kwargs_nescoil = quadcoil_kwargs_basic | {
    # The NESCOIL problem only contains the squared
    # flux objective. In QUADCOIL, this quantity is called
    # f_B.
    "objective_name": "f_B",
    # The NESCOIL problem is simple enough to need no normalization
    # constants. In the next example we will discuss how to choose
    # this constant.
    "objective_unit": None,
}

def solve_current_potential(eq):

    # Define a QuadcoilProxy with the simplest possible
    # signature.
    nescoil_objective = QuadcoilProxy(
        eq=eq,
        quadcoil_kwargs=quadcoil_kwargs_nescoil,
        vacuum=vacuum,
        # If you have additional filament/planar coils, put the CoilSet here.
        # field=[],
    )
    nescoil_objective.build()
    
    # Solving the NESCOIL problem
    out_dict_nescoil, qp_nescoil, dofs_nescoil, status_nescoil = (
        nescoil_objective.solve_quadcoil(
            # Like Objective.compute(), the xs of the equilibrium
            # must be passed in as the *arg.
            *nescoil_objective.xs(eq)
        )
    )
    
    # Defining problem
    quadcoil_kwargs = quadcoil_kwargs_basic | {
        "objective_name": "f_B",
        "objective_unit": f_B(qp_nescoil, dofs_nescoil),
        "constraint_name": ("f_max_B2_self",),
        "constraint_type": ("<=",),
        # Unit is not necessary under normalized=True. Under this mode,
        # they'll be automatically calculated from typical values in DESC.
        "constraint_unit": (B2_self_target,),
        "constraint_value": jnp.array([B2_self_target,]),
        "phi_init_with_nescoil": False,
    }
    quadcoil_fbe = QuadcoilFreeBoundaryError(
        eq=eq,
        quadcoil_kwargs=quadcoil_kwargs,
        enable_net_current_plasma=True,
        vacuum=vacuum,
        # WARNING: If changing this impacts the sln, then the unit 
        # generators may not be working!!!
        normalize=True, 
        normalize_target=False,
        weight=1,
    )
    quadcoil_fbe.build()
    return qp_nescoil, dofs_nescoil, quadcoil_fbe

In [ ]:
qp_nescoil_fbe, dofs_nescoil_fbe, quadcoil_fbe_fbe = solve_current_potential(eq_fbe)
qp_nescoil_qss, dofs_nescoil_qss, quadcoil_fbe_qss = solve_current_potential(eq_qss)
qp_nescoil_ori, dofs_nescoil_ori, quadcoil_fbe_ori = solve_current_potential(eq_ori)

In [ ]:
field_fbe = quadcoil_fbe_fbe.solve_quadcoil_surface_current(*quadcoil_fbe_fbe.xs(eq_fbe))
field_qss = quadcoil_fbe_fbe.solve_quadcoil_surface_current(*quadcoil_fbe_fbe.xs(eq_qss))
field_ori = quadcoil_fbe_fbe.solve_quadcoil_surface_current(*quadcoil_fbe_fbe.xs(eq_ori))

In [ ]:
_, qp_fbe, dofs_fbe, _ = quadcoil_fbe_fbe.solve_quadcoil(*quadcoil_fbe_fbe.xs(eq_fbe))
_, qp_qss, dofs_qss, _ = quadcoil_fbe_fbe.solve_quadcoil(*quadcoil_fbe_fbe.xs(eq_qss))
_, qp_ori, dofs_ori, _ = quadcoil_fbe_fbe.solve_quadcoil(*quadcoil_fbe_fbe.xs(eq_ori))

In [ ]:
print('Free boundary FBE:', quadcoil_fbe_fbe.compute_scalar(*quadcoil_fbe_fbe.xs(eq_fbe)))
print('          QSS FBE:', quadcoil_fbe_fbe.compute_scalar(*quadcoil_fbe_fbe.xs(eq_qss)))
print('   Reactor QA FBE:', quadcoil_fbe_fbe.compute_scalar(*quadcoil_fbe_fbe.xs(eq_ori)))

In [ ]:
def poincare(eq, field):
    grid_trace = LinearGrid(rho=np.linspace(0.0, 0.9, 10))
    r0 = eq.compute("R", grid=grid_trace)["R"]
    z0 = eq.compute("Z", grid=grid_trace)["Z"]
    time1 = time.time()
    fig, ax = poincare_plot(
        field, 
        R0=r0, Z0=z0,
        NFP=eq.NFP,
        size=0.5,
        ntransit=250
    )
    time2 = time.time()
    print('Poincare time:', time2-time1)

In [ ]:
poincare(eq_fbe, field_fbe)

In [ ]:
poincare(eq_qss, field_qss)

In [ ]:
poincare(eq_ori, field_ori)

In [ ]:
plt.style.use('default')
plt.figure(figsize=(18, 5))
plt.subplot(1,3,1)
plt.pcolor(np.sqrt(B2_self(qp_fbe, dofs_fbe)))
plt.colorbar(label=r'$B_{self}$ (T)')
plt.subplot(1,3,2)
plt.pcolor(np.sqrt(B2_self(qp_qss, dofs_qss)))
plt.colorbar(label=r'$B_{self}$ (T)')
plt.subplot(1,3,3)
plt.pcolor(np.sqrt(B2_self(qp_ori, dofs_ori)))
plt.colorbar(label=r'$B_{self}$ (T)')